# Zaskaleta AI Twin — AUTO v2
Автоматичний production notebook: Drive → 6 фото → master voice → behavior videos → Day 1–30 → сцени → lip-sync → фінальний ролик.


In [ ]:
import torch, subprocess, os, json
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Увімкніть T4 GPU у Runtime → Change runtime type")
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
!rm -rf /content/zaskaleta-ai-twin-colab
!git clone --depth 1 https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git /content/zaskaleta-ai-twin-colab

ROOT="/content/zaskaleta-ai-twin-colab"
WORKER=f"{ROOT}/worker"
PLAN=f"{ROOT}/content/monthly_plan_30_days.json"
DIALOGUES=f"{ROOT}/content/dialogue_overrides.json"
OUTFITS=f"{ROOT}/content/outfit_profiles.json"
CLONE_PROFILE=f"{ROOT}/content/clone_reference_profile.json"
MUSETALK="/content/MuseTalk"
VENV_DIR="/content/ai-twin-py311"
PYTHON_BIN=f"{VENV_DIR}/bin/python"

env=os.environ.copy()
env["APP_DIR"]=WORKER
env["MUSETALK_ROOT"]=MUSETALK
env["VENV_DIR"]=VENV_DIR
r=subprocess.run(["bash",f"{WORKER}/install_gpu_engines.sh"],env=env)
if r.returncode != 0:
    raise RuntimeError(f"Installer failed: {r.returncode}")
print("✅ Production code + engines ready")


## 1. Google Drive — автоматично знайти базу клона


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

ASSET_MAP="/content/clone_assets.json"
subprocess.run([
    PYTHON_BIN,
    f"{WORKER}/locate_clone_assets.py",
    "--mydrive","/content/drive/MyDrive",
    "--profile",CLONE_PROFILE,
    "--output",ASSET_MAP
],check=True)

assets=json.loads(Path(ASSET_MAP).read_text(encoding="utf-8"))
BASE=Path(assets["base_dir"])
MASTER_PHOTOS=assets["master_photos"]
VOICE=assets["master_voice"]
BEHAVIOR_VIDEOS=[x["path"] for x in assets["behavior_videos"]]

print("📁 BASE:",BASE)
print("🖼️ MASTER PHOTOS:",len(MASTER_PHOTOS))
print("🎙️ MASTER VOICE:",Path(VOICE).name)
print("🎥 BEHAVIOR VIDEOS:",len(BEHAVIOR_VIDEOS))
for p in BEHAVIOR_VIDEOS:
    print(" •",Path(p).name)


## 2. Вибрати день 1–30


In [ ]:
import ipywidgets as widgets
from IPython.display import display

plan_text=Path(PLAN).read_text(encoding="utf-8")
plan_text=plan_text.replace('Не кожне "ні" — слабкість.', 'Не кожне «ні» — слабкість.')
plan_text=plan_text.replace('Сказати "я зроблю" займає секунду.', 'Сказати «я зроблю» займає секунду.')
plan=json.loads(plan_text)
dialogue_plan=json.loads(Path(DIALOGUES).read_text(encoding="utf-8"))
dialogue_days={int(k) for k in dialogue_plan.get("days",{}).keys()}

day_pick=widgets.Dropdown(
    options=[
        (f"Day {d['day']:02d} — {d['title']} — {d['city']}" + (" 🗣️" if d["day"] in dialogue_days else ""), d["day"])
        for d in plan["days"]
    ],
    description="DAY:"
)
display(day_pick)


In [ ]:
DAY=int(day_pick.value)
DAYDIR=BASE/"episodes"/f"Day_{DAY:02d}"
DAYDIR.mkdir(parents=True,exist_ok=True)

subprocess.run([
    PYTHON_BIN,f"{WORKER}/prepare_daily_episode.py",
    "--plan",PLAN,
    "--day",str(DAY),
    "--output-dir",str(DAYDIR),
    "--dialogues",DIALOGUES,
    "--outfits",OUTFITS,
    "--clone-profile",CLONE_PROFILE
],check=True)

episode=json.loads((DAYDIR/"episode.json").read_text(encoding="utf-8"))
print("🎬",episode["title"])
print("📍",episode["city"],"—",episode["location"])
print("👕",episode["outfit"])
print("🗣️ Dialogue:","YES" if episode.get("dialogue") else "NO")
print("🎭 REALISM LOCK: ON")


## 3. Голоси та покадрова мова


In [ ]:
DIALOGUE_AUDIO_DIR=DAYDIR/"dialogue_audio"
subprocess.run([
    PYTHON_BIN,f"{WORKER}/generate_dialogue_audio.py",
    "--episode",str(DAYDIR/"episode.json"),
    "--master-voice",VOICE,
    "--worker-dir",WORKER,
    "--python-bin",PYTHON_BIN,
    "--output-dir",str(DIALOGUE_AUDIO_DIR)
],check=True)

SCENE_SPEECH_DIR=DAYDIR/"scene_speech"
subprocess.run([
    PYTHON_BIN,f"{WORKER}/generate_scene_speech.py",
    "--episode",str(DAYDIR/"episode.json"),
    "--manifest",str(DAYDIR/"scene_prompts.json"),
    "--master-voice",VOICE,
    "--output-dir",str(SCENE_SPEECH_DIR)
],check=True)
print("✅ Speech pack ready")


## 4. Згенерувати 8 фотореалістичних сцен


In [ ]:
KEYFRAMES=DAYDIR/"keyframes"
subprocess.run([
    PYTHON_BIN,f"{WORKER}/generate_scene_keyframes.py",
    "--manifest",str(DAYDIR/"scene_prompts.json"),
    "--photos",*MASTER_PHOTOS,
    "--output-dir",str(KEYFRAMES),
    "--seed","9969"
],check=True)
print("✅ Keyframes ready")

from IPython.display import Image as IPImage, display
for i in range(1,9):
    p=KEYFRAMES/f"scene_{i:02d}.png"
    if p.is_file():
        display(IPImage(filename=str(p),width=260))


## 5. Анімація + lip-sync


In [ ]:
ANIMATED=DAYDIR/"animated"
subprocess.run([
    PYTHON_BIN,f"{WORKER}/animate_scene_keyframes.py",
    "--manifest",str(DAYDIR/"scene_prompts.json"),
    "--image-dir",str(KEYFRAMES),
    "--output-dir",str(ANIMATED)
],check=True)

subprocess.run([
    PYTHON_BIN,f"{WORKER}/render_scene_clips.py",
    "--manifest",str(DAYDIR/"scene_prompts.json"),
    "--keyframes",str(KEYFRAMES),
    "--scene-speech",str(SCENE_SPEECH_DIR/"scene_speech_manifest.json"),
    "--dialogue-audio-dir",str(DIALOGUE_AUDIO_DIR),
    "--animated-dir",str(ANIMATED),
    "--worker-dir",WORKER,
    "--python-bin",PYTHON_BIN,
    "--output-dir",str(DAYDIR)
],check=True)
print("✅ 8 final scene clips ready")


## 6. Фінальна короткометражка 60–90 секунд


In [ ]:
FINAL=str(DAYDIR/f"Day_{DAY:02d}_FINAL_9x16.mp4")
subprocess.run([
    PYTHON_BIN,f"{WORKER}/assemble_daily_episode.py",
    "--episode-dir",str(DAYDIR),
    "--output",FINAL
],check=True)

print("✅ FINAL:",FINAL)
from IPython.display import Video,display
display(Video(FINAL,embed=True,width=360))
